# FGNN2 on the Circuitree v2 dataset

This notebook tests the FGNN2 model on the Circuitree v2 quiz from my lead.

1. **Setup**: install everything (run every time you open the notebook)
2. **Code files**: saves the 3 scripts (just run them)
3. **Fingerprints + scores**: FGNN2 takes the quiz, the grader scores it
4. **Top 20 + CEC**: for one query circuit, FGNN2's 20 closest picks, each checked by ABC

Before starting: **Runtime → Change runtime type → T4 GPU**.

## 1. Setup

Put the dataset zip in Google Drive first. If its name or folder is different, change `ZIP` below.

In [ ]:
ZIP = "/content/drive/MyDrive/Circuitree/circuitree_v2__1_.zip"

In [ ]:
# connect Google Drive (click Allow when it asks)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# FGNN2 code + a Python 3.12 "room" with PyTorch 2.4 and DGL 2.4 (takes ~3 min)
!git clone -q https://github.com/ZeayW/FGNN2.git
!pip install -q uv
!uv venv /content/fgnn_env --python 3.12 --seed
!/content/fgnn_env/bin/pip install -q "numpy<2" torch==2.4.0 --index-url https://download.pytorch.org/whl/cu124 --extra-index-url https://pypi.org/simple
!/content/fgnn_env/bin/pip install -q packaging pydantic pyyaml pandas psutil tqdm requests networkx scipy
!/content/fgnn_env/bin/pip install -q "dgl==2.4.0+cu124" -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
!/content/fgnn_env/bin/pip install -q "torchdata==0.8.0" --no-deps

In [ ]:
# FGNN2's trained weights, ABC (for CEC), and the dataset
!mkdir -p FGNN2/checkpoints/example
!gdown -q 1NGAJRedx040A4EeN9mALXsly87TRseRF -O FGNN2/checkpoints/example/weight.pth
!apt-get -qq install -y yosys > /dev/null
!unzip -q -o "{ZIP}" -d /content/

In [ ]:
# check: should print 2.4.0+cu124, 2.4.0+cu124, True, then the path to yosys-abc and the dataset files
!/content/fgnn_env/bin/python -c "import torch, dgl; print(torch.__version__); print(dgl.__version__); print(torch.cuda.is_available())" 2>/dev/null
!which yosys-abc
!ls /content/circuitree_v2

## 2. Code files

Each cell saves one script. No need to read or change them, just run all six.

In [ ]:
%%writefile /content/FGNN2/src/eval_retrieval.py
"""
eval_retrieval.py  --  measure FGNN2 on circuit retrieval
Metrics: retrieval@k (hit@k), MRR, top-1 accuracy, and timing.

Put this file in FGNN2/src/ (it imports FunctionConv.py).

Expected data layout (one folder per source circuit, variants inside):
    bench_root/
        c432/   c432_v0.bench  c432_v1.bench ...   <- all variants of c432
        c880/   c880_v0.bench  ...
Two circuits "match" if they come from the same folder.

Example:
    python eval_retrieval.py --bench_root ../data/iscas_variants \
        --weights ../checkpoints/example/weight.pth --flag_inv --ks 1 5 10
"""
import argparse, os, time, json
import numpy as np


# ---------------------------------------------------------------- parsing
def parse_bench(path):
    """Parse an ABC-style AIG .bench (INPUT/OUTPUT/AND/NOT/BUFF, multi-output).
    NOT gates are folded into inverted edges (edge flag r=1), same idea as
    pretrain_dataparser.py, but this version supports many outputs."""
    inputs, outputs, defs = [], [], {}
    with open(path) as f:
        for raw in f:
            line = raw.split('#')[0].strip()
            if not line:
                continue
            up = line.upper()
            if up.startswith('INPUT('):
                inputs.append(line[line.find('(') + 1:line.rfind(')')].strip())
            elif up.startswith('OUTPUT('):
                outputs.append(line[line.find('(') + 1:line.rfind(')')].strip())
            elif '=' in line:
                lhs, rhs = [s.strip() for s in line.split('=', 1)]
                if '(' in rhs:
                    gate = rhs[:rhs.find('(')].strip().upper()
                    args = [a.strip() for a in rhs[rhs.find('(') + 1:rhs.rfind(')')].split(',') if a.strip()]
                else:                      # e.g. "n0 = vdd" / "n0 = gnd"
                    gate, args = rhs.strip().upper(), []
                defs[lhs] = (gate, args)

    def resolve(sig, depth=0):
        """Follow NOT/BUFF chains -> (real node name, inverted?)"""
        inv = False
        while sig in defs and defs[sig][0] in ('NOT', 'BUFF', 'BUF'):
            g, a = defs[sig]
            if g == 'NOT':
                inv = not inv
            sig = a[0]
        return sig, inv

    nid = {}
    def get_id(name):
        if name not in nid:
            nid[name] = len(nid)
        return nid[name]

    for pi in inputs:
        get_id(pi)
    src, dst, r = [], [], []
    for name, (gate, args) in defs.items():
        if gate == 'AND':
            d = get_id(name)
            for a in args:
                s, inv = resolve(a)
                src.append(get_id(s)); dst.append(d); r.append(int(inv))
        elif gate in ('NOT', 'BUFF', 'BUF'):
            continue
        elif gate in ('VDD', 'GND', 'CONST0', 'CONST1'):
            get_id(name)              # constant: source node, like a PI
        else:
            raise ValueError(f'{path}: gate {gate} is not AIG. Run "strash" in ABC first.')

    po_nids, po_inv = [], []
    for po in outputs:
        s, inv = resolve(po)
        po_nids.append(get_id(s)); po_inv.append(int(inv))

    return dict(n_nodes=len(nid), src=src, dst=dst, r=r,
                po_nids=po_nids, po_inv=po_inv,
                n_pi=len(inputs), n_and=sum(1 for g, _ in defs.values() if g == 'AND'))


def build_graph(p):
    import torch as th, dgl
    g = dgl.graph((th.tensor(p['src'], dtype=th.long), th.tensor(p['dst'], dtype=th.long)),
                  num_nodes=p['n_nodes'])
    g.edata['r'] = th.tensor(p['r'], dtype=th.long)
    g.ndata['inv'] = th.zeros(p['n_nodes'])      # PO inversion handled after the GNN
    # input position: input i (the i-th INPUT line, e.g. x0, x1, ...) gets i+1; every other node gets 0.
    # Only used by models that have a pos_emb table (Experiment B); ignored otherwise.
    pos = th.zeros(p['n_nodes'], dtype=th.long)
    pos[:p['n_pi']] = th.arange(1, p['n_pi'] + 1)
    g.ndata['pos'] = pos
    return g


# ---------------------------------------------------------------- embedding
def embed_batch(model, graphs, po_lists, po_inv_lists, device):
    """Embed many circuits in ONE forward pass (dgl.batch).
    Returns [num_graphs, 2*hidden] = concat(mean, max) over PO embeddings,
    the same readout train_global.py uses."""
    import torch as th, dgl
    bg = dgl.batch(graphs).to(device)
    h0 = th.ones((bg.num_nodes(), model.hidden_dim), device=device)
    pos_emb = getattr(model, 'pos_emb', None)
    if pos_emb is not None:                                   # Experiment B: each input gets its own start
        p = bg.ndata['pos']
        p = th.where(p < pos_emb.num_embeddings, p, th.zeros_like(p))
        h0 = h0 + pos_emb(p)                                  # row 0 is always zero -> gates unchanged
    bg.ndata['h'] = h0
    topo = [l.to(device) for l in dgl.topological_nodes_generator(bg)]
    use_proj = model.flag_proj
    model.set_flag_proj(False)                                # project only after PO inversion
    h = model(bg, topo, None, flag_usage='global')            # all node embeddings
    model.set_flag_proj(use_proj)

    offs = th.cumsum(th.tensor([0] + [g.num_nodes() for g in graphs[:-1]]), 0)
    po_idx, seg, inv = [], [], []
    for k, (pl, il) in enumerate(zip(po_lists, po_inv_lists)):
        po_idx += [int(offs[k]) + p for p in pl]; seg += [k] * len(pl); inv += il
    po_idx = th.tensor(po_idx, device=device); seg = th.tensor(seg, device=device)
    inv = th.tensor(inv, device=device, dtype=th.bool)

    po_h = h[po_idx].clone()
    if model.flag_inv and inv.any():
        po_h[inv] = model.func_inv(po_h[inv])                 # inverted outputs
    if use_proj:
        po_h = model.proj_head(po_h)

    B, D = len(graphs), po_h.shape[1]
    cnt = th.zeros(B, device=device).index_add_(0, seg, th.ones_like(seg, dtype=th.float))
    mean = th.zeros(B, D, device=device).index_add_(0, seg, po_h) / cnt.clamp(min=1)[:, None]
    mx = th.full((B, D), -1e9, device=device).scatter_reduce(
        0, seg[:, None].expand(-1, D), po_h, reduce='amax', include_self=True)
    return th.cat([mean, mx], 1)


# ---------------------------------------------------------------- metrics
def retrieval_metrics(emb, groups, ks=(1, 5, 10)):
    """Leave-one-out retrieval. Every circuit is a query; the rest is the gallery.
    hit@k = was at least one same-group circuit in the top k?
    MRR   = average of 1/(rank of first same-group circuit)
    acc   = top-1 nearest neighbour has the right group (== hit@1)"""
    emb = np.asarray(emb, dtype=np.float64)
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
    groups = np.asarray(groups)
    sim = emb @ emb.T
    np.fill_diagonal(sim, -np.inf)
    order = np.argsort(-sim, axis=1)
    rel = groups[order] == groups[:, None]
    has = rel.any(1)                                  # skip groups of size 1
    first = rel[has].argmax(1) + 1                    # 1-based rank
    out = {f'hit@{k}': float((first <= k).mean()) for k in ks}
    out['MRR'] = float((1.0 / first).mean())
    out['top1_acc'] = out.get('hit@1', float((first == 1).mean()))
    out['mean_rank'] = float(first.mean())
    out['num_queries'] = int(has.sum())
    return out


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--bench_root', required=True)
    ap.add_argument('--weights', default=None, help='pretrained FuncConv state_dict (.pth)')
    ap.add_argument('--hidden_dim', type=int, default=256)
    ap.add_argument('--out_dim', type=int, default=256)
    ap.add_argument('--flag_inv', action='store_true')
    ap.add_argument('--flag_proj', action='store_true')
    ap.add_argument('--batch_size', type=int, default=64)
    ap.add_argument('--ks', type=int, nargs='+', default=[1, 5, 10])
    ap.add_argument('--gpu', type=int, default=0)
    ap.add_argument('--out_json', default=None)
    ap.add_argument('--show_misses', action='store_true',
                    help='print every circuit whose #1 pick was not a twin')
    a = ap.parse_args()

    import torch as th
    from FunctionConv import FuncConv
    device = th.device(f'cuda:{a.gpu}' if th.cuda.is_available() else 'cpu')
    sync = (lambda: th.cuda.synchronize()) if device.type == 'cuda' else (lambda: None)

    model = FuncConv(a.hidden_dim, a.out_dim, flag_proj=a.flag_proj, flag_inv=a.flag_inv).to(device)
    if a.weights:
        model.load_state_dict(th.load(a.weights, map_location=device, weights_only=True))
    else:
        print('WARNING: no --weights, using a random untrained model (useful as a baseline)')
    model.eval()

    # 1) parse
    t0 = time.perf_counter()
    items = []
    for grp in sorted(os.listdir(a.bench_root)):
        gdir = os.path.join(a.bench_root, grp)
        if not os.path.isdir(gdir):
            continue
        for fn in sorted(os.listdir(gdir)):
            if fn.endswith('.bench'):
                items.append((grp, fn, parse_bench(os.path.join(gdir, fn))))
    t_parse = time.perf_counter() - t0
    print(f'parsed {len(items)} circuits from {len({i[0] for i in items})} groups')

    # 2) build graphs
    t0 = time.perf_counter()
    graphs = [build_graph(p) for _, _, p in items]
    t_build = time.perf_counter() - t0

    # 3) embed (timed)
    embs = []
    with th.no_grad():
        embed_batch(model, graphs[:2], [p['po_nids'] for *_, p in items[:2]],
                    [p['po_inv'] for *_, p in items[:2]], device)      # warm-up
        sync(); t0 = time.perf_counter()
        for s in range(0, len(graphs), a.batch_size):
            chunk = items[s:s + a.batch_size]
            embs.append(embed_batch(model, graphs[s:s + a.batch_size],
                                    [p['po_nids'] for *_, p in chunk],
                                    [p['po_inv'] for *_, p in chunk], device).cpu())
        sync(); t_embed = time.perf_counter() - t0
    emb = th.cat(embs).numpy()

    # 4) search + metrics (timed)
    t0 = time.perf_counter()
    res = retrieval_metrics(emb, [g for g, _, _ in items], a.ks)
    t_search = time.perf_counter() - t0

    n = len(items)
    res.update({
        'num_circuits': n, 'device': str(device),
        'time_parse_s': t_parse, 'time_build_s': t_build,
        'time_embed_s': t_embed, 'ms_per_circuit_embed': 1000 * t_embed / n,
        'time_search_all_queries_s': t_search,
        'avg_and_nodes': float(np.mean([p['n_and'] for *_, p in items])),
    })
    for k, v in res.items():
        print(f'{k:>28}: {v:.4f}' if isinstance(v, float) else f'{k:>28}: {v}')
    if a.show_misses:
        e = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
        sim = e @ e.T
        np.fill_diagonal(sim, -np.inf)
        top = sim.argmax(1)
        print('\nMISSES (circuit -> its #1 pick, which was NOT a twin):')
        found = False
        for i in range(n):
            if items[top[i]][0] != items[i][0]:
                found = True
                print(f'  {items[i][1]:>20} -> {items[top[i]][1]:<20} similarity {sim[i, top[i]]:.4f}')
        if not found:
            print('  none!')
    if a.out_json:
        with open(a.out_json, 'w') as f:
            json.dump(res, f, indent=2)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/FGNN2/src/embed_circuitree.py
"""
embed_circuitree.py -- make FGNN2 fingerprints for the Circuitree v2 dataset.

Put this file in FGNN2/src/ next to eval_retrieval.py (it reuses its parser and batching).
It writes an .npz with 'sample_ids' and 'embeddings', which the dataset's own
evaluate.py scores.

Example:
    python embed_circuitree.py --data_root /content/circuitree_v2 --track real --split val \
        --weights ../checkpoints/example/weight.pth --flag_inv --out /content/fgnn2_real_val.npz
"""
import argparse, csv, os, time
import numpy as np
from eval_retrieval import parse_bench, build_graph, embed_batch


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data_root', required=True, help='the circuitree_v2 folder')
    ap.add_argument('--track', choices=['real', 'synthetic'], required=True)
    ap.add_argument('--split', choices=['train', 'val', 'test'], default='val')
    ap.add_argument('--weights', default=None, help='leave out = untrained (random) FGNN2')
    ap.add_argument('--hidden_dim', type=int, default=256)
    ap.add_argument('--flag_inv', action='store_true')
    ap.add_argument('--flag_proj', action='store_true')
    ap.add_argument('--positions', action='store_true', help='model has input positions (Experiment B/C weights)')
    ap.add_argument('--input_count', action='store_true', help='apply the input-count rule to the fingerprints')
    ap.add_argument('--batch_size', type=int, default=128)
    ap.add_argument('--gpu', type=int, default=0)
    ap.add_argument('--out', required=True)
    a = ap.parse_args()

    import torch as th
    from fgnn2_pos import make_model, add_input_count
    device = th.device(f'cuda:{a.gpu}' if th.cuda.is_available() else 'cpu')
    sync = (lambda: th.cuda.synchronize()) if device.type == 'cuda' else (lambda: None)

    model = make_model(a.hidden_dim, positions=a.positions, weights=a.weights, device=device)
    model.set_flag_proj(a.flag_proj); model.flag_inv = a.flag_inv
    if not a.weights:
        print('WARNING: no --weights, using an untrained (random) FGNN2')
    model.eval()

    # every circuit of this track + split (near-miss samples included, evaluate.py needs them)
    with open(os.path.join(a.data_root, 'samples.csv'), newline='') as f:
        rows = [r for r in csv.DictReader(f) if r['track'] == a.track and r['split'] == a.split]
    # circuits of similar depth go in the same batch -> fewer wasted steps
    rows.sort(key=lambda r: int(r['depth']))
    print(f'{len(rows)} circuits ({a.track} {a.split})')

    t0 = time.perf_counter()
    parsed = [parse_bench(os.path.join(a.data_root, r['file'])) for r in rows]
    graphs = [build_graph(p) for p in parsed]
    t_load = time.perf_counter() - t0

    embs = []
    with th.no_grad():
        embed_batch(model, graphs[:2], [p['po_nids'] for p in parsed[:2]],
                    [p['po_inv'] for p in parsed[:2]], device)            # warm-up
        sync(); t0 = time.perf_counter()
        for s in range(0, len(graphs), a.batch_size):
            e = embed_batch(model, graphs[s:s + a.batch_size],
                            [p['po_nids'] for p in parsed[s:s + a.batch_size]],
                            [p['po_inv'] for p in parsed[s:s + a.batch_size]], device)
            embs.append(e.cpu().numpy())
            if (s // a.batch_size) % 10 == 0:
                print(f'  {min(s + a.batch_size, len(graphs))}/{len(graphs)} done')
        sync(); t_embed = time.perf_counter() - t0

    E = np.concatenate(embs)
    if a.input_count:
        E = add_input_count(E, [p['n_pi'] for p in parsed])
        print('input-count rule applied')
    np.savez(a.out, sample_ids=np.array([r['sample_id'] for r in rows]), embeddings=E)
    print(f'saved {a.out}')
    print(f'time to read files: {t_load:.1f} s')
    print(f'time to embed:      {t_embed:.1f} s  ({1000 * t_embed / len(rows):.2f} ms per circuit)')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/top20_cec.py
"""
top20_cec.py -- for one query circuit, show the top-k most similar circuits by FGNN2
fingerprint, and check each one with ABC's CEC (combinational equivalence checking).

Needs: the .npz from embed_circuitree.py, the circuitree_v2 folder, and yosys-abc
(Colab: !apt-get -qq install -y yosys). Only numpy is needed, no PyTorch.

Example:
    python top20_cec.py --data_root /content/circuitree_v2 --emb /content/fgnn2_real_val.npz \
        --track real --split val --query r027344 --k 20 --out /content/top20_r027344.csv
Leave out --query to pick a random evaluation query (use --seed to change which one).
"""
import argparse, csv, os, random, subprocess, time
import numpy as np


def add_input_count(E, num_inputs, weight=10.0, max_inputs=256):
    """Input-count rule (same as fgnn2_pos.add_input_count): circuits with a different number of
    inputs can never look close."""
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    oh = np.zeros((len(E), max_inputs + 1))
    oh[np.arange(len(E)), np.minimum(np.asarray(num_inputs), max_inputs)] = weight
    return np.hstack([E, oh])


def run_cec(abc, a, b):
    """Return (result, seconds). result: EQUIVALENT / NOT EQUIVALENT / DIFFERENT #INPUTS / ERROR"""
    t = time.perf_counter()
    out = subprocess.run([abc, '-q', f'cec {a} {b}'], capture_output=True, text=True, timeout=120)
    sec = time.perf_counter() - t
    text = out.stdout + out.stderr
    if 'NOT EQUIVALENT' in text:
        return 'NOT EQUIVALENT', sec
    if 'are equivalent' in text:
        return 'EQUIVALENT', sec
    if 'different number of primary inputs' in text:
        return 'DIFFERENT #INPUTS', sec
    return 'ERROR', sec


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data_root', required=True)
    ap.add_argument('--emb', required=True, help='.npz from embed_circuitree.py')
    ap.add_argument('--track', default='real')
    ap.add_argument('--split', default='val')
    ap.add_argument('--query', default=None, help='sample_id; leave out for a random query')
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--k', type=int, default=20)
    ap.add_argument('--abc', default='yosys-abc')
    ap.add_argument('--out', default=None, help='optional .csv to save the table')
    ap.add_argument('--input_count', action='store_true', help='apply the input-count rule first')
    a = ap.parse_args()

    with open(os.path.join(a.data_root, 'samples.csv'), newline='') as f:
        info = {r['sample_id']: r for r in csv.DictReader(f)}

    z = np.load(a.emb)
    ids = z['sample_ids'].tolist()
    emb = z['embeddings'].astype(np.float64)
    if a.input_count:
        emb = add_input_count(emb, [int(info[s]['num_inputs']) for s in ids])
    emb /= np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12

    q = a.query
    if q is None:
        qfile = os.path.join(a.data_root, 'eval', f'{a.track}_{a.split}_queries.csv')
        with open(qfile, newline='') as f:
            q = random.Random(a.seed).choice([r['query_id'] for r in csv.DictReader(f)])
    qi = ids.index(q)
    qrow = info[q]

    sim = emb @ emb[qi]
    # same ranking rule as strict_eval.py: highest similarity first, ties broken by sample_id
    order = np.lexsort((np.array(ids), -sim))
    top = order[order != qi][:a.k]                      # don't return the query itself

    total_twins = sum(1 for s in ids if s != q and info[s]['function_id'] == qrow['function_id'])
    qpath = os.path.join(a.data_root, qrow['file'])
    print(f"QUERY {q}: {qrow['num_inputs']} inputs, {qrow['and_count']} ANDs, depth {qrow['depth']}, "
          f"from {qrow['source_circuit']}")
    print(f"gallery = all {len(ids) - 1} other circuits in {a.track} {a.split}; "
          f"true twins in gallery: {total_twins}\n")

    header = ['rank', 'sample_id', 'similarity', 'inputs', 'ANDs', 'depth', 'role', 'label_twin', 'CEC', 'cec_s']
    print(f"{'rank':>4}  {'sample_id':<9} {'sim':>7} {'in':>4} {'ANDs':>5} {'dep':>4}  {'role':<9} "
          f"{'twin?':<5}  CEC")
    table, n_eq, agree, t_cec = [], 0, 0, 0.0
    for rank, j in enumerate(top, 1):
        s = ids[j]; r = info[s]
        label = 'yes' if r['function_id'] == qrow['function_id'] else 'no'
        cec, sec = run_cec(a.abc, qpath, os.path.join(a.data_root, r['file']))
        t_cec += sec
        n_eq += cec == 'EQUIVALENT'
        agree += (cec == 'EQUIVALENT') == (label == 'yes')
        table.append([rank, s, f'{sim[j]:.4f}', r['num_inputs'], r['and_count'], r['depth'],
                      r['role'], label, cec, f'{sec:.3f}'])
        print(f"{rank:>4}  {s:<9} {sim[j]:>7.4f} {r['num_inputs']:>4} {r['and_count']:>5} {r['depth']:>4}  "
              f"{r['role']:<9} {label:<5}  {cec}")

    twin_ranks = [row[0] for row in table if row[7] == 'yes']
    strict_mrr = f'{1 / twin_ranks[0]:.3f}' if twin_ranks else f'< {1 / (a.k + 1):.3f}'
    print(f"\nthis query, strict scores (copies count as wrong): Hit@1 {int(bool(twin_ranks) and twin_ranks[0] == 1)} | "
          f"MRR {strict_mrr} | twins in top {a.k}: {len(twin_ranks)}/{total_twins}")
    print(f"CEC says {n_eq} of the top {a.k} are truly equivalent "
          f"(out of {total_twins} twins that exist).")
    print(f"CEC agrees with the dataset labels on {agree}/{a.k}.  Total CEC time: {t_cec:.2f} s")
    if a.out:
        with open(a.out, 'w', newline='') as f:
            w = csv.writer(f); w.writerow(header); w.writerows(table)
        print(f'saved {a.out}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/FGNN2/src/finetune_fgnn2.py
"""
finetune_fgnn2.py -- fine-tune FGNN2 on the Circuitree v2 train split.

Experiment A: same model, only the weights get more practice.
Experiment B (--positions): each input also gets its own learned starting fingerprint.
Experiment C: no training; Exp B's weights + the input-count rule (fgnn2_pos.add_input_count).
Put this file in FGNN2/src/ next to eval_retrieval.py.

How one training step works ("pull and push"):
  * pick a group of different functions (families) with similar depth
  * take 2 twins (views) from each family            -> PULL their fingerprints together
  * every other circuit in the step is a negative     -> PUSH them apart
  * near-miss imposters of those families are extra negatives
Every --eval_every steps it scores the val split with the dataset's own evaluate.py
and saves the best weights (by val hard Hit@1). The test split is never touched.

Example:
    python finetune_fgnn2.py --data_root /content/circuitree_v2 \
        --weights ../checkpoints/example/weight.pth --steps 2000 --out_dir /content/expA
"""
import argparse, csv, os, random, sys, time
import numpy as np
from eval_retrieval import parse_bench, build_graph, embed_batch


def load_split(root, track, split):
    with open(os.path.join(root, 'samples.csv'), newline='') as f:
        rows = [r for r in csv.DictReader(f) if r['track'] == track and r['split'] == split]
    t = time.perf_counter()
    parsed = [parse_bench(os.path.join(root, r['file'])) for r in rows]
    graphs = [build_graph(p) for p in parsed]
    print(f'  loaded {len(rows)} {split} circuits in {time.perf_counter() - t:.0f} s')
    return rows, parsed, graphs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data_root', required=True)
    ap.add_argument('--track', default='real')
    ap.add_argument('--weights', default=None, help='start from these weights (leave out = from scratch)')
    ap.add_argument('--hidden_dim', type=int, default=256)
    ap.add_argument('--steps', type=int, default=2000)
    ap.add_argument('--batch_functions', type=int, default=64, help='families per step')
    ap.add_argument('--node_budget', type=int, default=60000, help='max AND gates per step (avoids running out of GPU memory)')
    ap.add_argument('--lr', type=float, default=1e-4)
    ap.add_argument('--temperature', type=float, default=0.1)
    ap.add_argument('--no_near_miss', action='store_true', help="don't use imposters as extra negatives")
    ap.add_argument('--depth_bins', type=int, default=8)
    ap.add_argument('--eval_every', type=int, default=250)
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--positions', action='store_true', help='Experiment B: give each input its own position')
    ap.add_argument('--pos_lr', type=float, default=1e-3, help='learning rate for the new position vectors')
    ap.add_argument('--select', choices=['hard', 'full_rule'], default='hard',
                    help='which val score picks the best weights (full_rule = full Hit@1 with the input-count rule)')
    ap.add_argument('--out_dir', required=True)
    a = ap.parse_args()

    import torch as th
    import torch.nn.functional as F
    from fgnn2_pos import make_model, add_input_count
    sys.path.insert(0, a.data_root)
    import evaluate as ev

    random.seed(a.seed); np.random.seed(a.seed); th.manual_seed(a.seed)
    os.makedirs(a.out_dir, exist_ok=True)
    device = th.device('cuda' if th.cuda.is_available() else 'cpu')

    model = make_model(a.hidden_dim, positions=a.positions, weights=a.weights, device=device)
    print('starting from', 'weights ' + a.weights if a.weights else 'random weights (from scratch)',
          '| input positions:', 'ON' if a.positions else 'off')

    print('loading data (takes a minute or two)...')
    tr_rows, tr_parsed, tr_graphs = load_split(a.data_root, a.track, 'train')
    va_rows, va_parsed, va_graphs = load_split(a.data_root, a.track, 'val')
    va_order = sorted(range(len(va_rows)), key=lambda i: int(va_rows[i]['depth']))

    # ---- group the training circuits into families
    by_fn, mutants_of, pg_of, depth_of, size_of = {}, {}, {}, {}, {}
    for i, r in enumerate(tr_rows):
        if r['role'] == 'view':
            by_fn.setdefault(r['function_id'], []).append(i)
            pg_of[r['function_id']] = r['perm_group_id']
        else:
            mutants_of.setdefault(r['source_function_id'], []).append(i)
    functions = [f for f, v in by_fn.items() if len(v) >= 2]
    for f in functions:
        depth_of[f] = float(np.median([int(tr_rows[i]['depth']) for i in by_fn[f]]))
        size_of[f] = float(np.mean([int(tr_rows[i]['and_count']) for i in by_fn[f]]))
    # depth bins: each step uses families of similar depth (faster + harder negatives)
    edges = np.quantile([depth_of[f] for f in functions], np.linspace(0, 1, a.depth_bins + 1))
    bins = [[] for _ in range(a.depth_bins)]
    for f in functions:
        b = min(int(np.searchsorted(edges, depth_of[f], side='right')) - 1, a.depth_bins - 1)
        bins[max(b, 0)].append(f)
    bins = [b for b in bins if len(b) >= 8]
    print(f'train: {len(functions)} families, {sum(len(v) for v in mutants_of.values())} imposters, '
          f'{len(bins)} depth bins')

    def batch_embed(idx_list, graphs, parsed):
        return embed_batch(model, [graphs[i] for i in idx_list], [parsed[i]['po_nids'] for i in idx_list],
                           [parsed[i]['po_inv'] for i in idx_list], device)

    def score_val():
        model.eval()
        embs = {}
        with th.no_grad():
            for s in range(0, len(va_order), 128):
                chunk = va_order[s:s + 128]
                e = batch_embed(chunk, va_graphs, va_parsed).cpu().numpy()
                for i, v in zip(chunk, e):
                    embs[va_rows[i]['sample_id']] = v
        model.train()
        hard = ev.evaluate(embs, a.track, 'val', 'hard', root=a.data_root)
        full = ev.evaluate(embs, a.track, 'val', 'full', root=a.data_root)
        ids = list(embs)
        ruled = add_input_count(np.stack([embs[s] for s in ids]), [n_in[s] for s in ids])
        full_rule = ev.evaluate(dict(zip(ids, ruled)), a.track, 'val', 'full', root=a.data_root)
        return hard, full, full_rule

    n_in = {va_rows[i]['sample_id']: va_parsed[i]['n_pi'] for i in range(len(va_rows))}
    pick = lambda hard, full_rule: hard['hit1'] if a.select == 'hard' else full_rule['hit1']

    def info_nce(za, zb, extra):
        logits = za @ zb.t()
        if extra is not None:
            logits = th.cat([logits, za @ extra.t()], 1)
        return F.cross_entropy(logits / a.temperature, th.arange(len(za), device=za.device))

    pos_params = list(model.pos_emb.parameters()) if a.positions else []
    other = [p for p in model.parameters() if all(p is not q for q in pos_params)]
    groups = [{'params': other, 'lr': a.lr}]
    if pos_params:
        groups.append({'params': pos_params, 'lr': a.pos_lr})   # brand-new, so it learns faster
    opt = th.optim.Adam(groups)
    log_path = os.path.join(a.out_dir, 'log.csv')
    log = open(log_path, 'w', newline='')
    logw = csv.writer(log)
    logw.writerow(['step', 'avg_loss', 'val_hard_hit1', 'val_hard_mrr', 'val_full_hit1',
                   'val_full_mrr', 'val_full_rule_hit1', 'val_full_rule_mrr', 'minutes'])

    hard, full, full_rule = score_val()
    best = pick(hard, full_rule)
    print(f"step 0 (before training): val hard Hit@1 {hard['hit1']:.4f} | full Hit@1 {full['hit1']:.4f} "
          f"| full + input-count rule {full_rule['hit1']:.4f}")
    logw.writerow([0, '', hard['hit1'], hard['mrr'], full['hit1'], full['mrr'],
                   full_rule['hit1'], full_rule['mrr'], 0]); log.flush()
    th.save(model.state_dict(), os.path.join(a.out_dir, 'best.pth'))

    t_start = time.perf_counter()
    losses = []
    model.train()
    for step in range(1, a.steps + 1):
        fams = random.choice(bins)[:]
        random.shuffle(fams)
        avg_size = np.mean([size_of[f] for f in fams])
        B = int(max(8, min(a.batch_functions, a.node_budget // (3 * max(avg_size, 1)))))
        chosen, used_pg = [], set()
        for f in fams:
            # without positions, swapped-input families look identical to FGNN2, so keep only one
            # per step. With positions the model CAN tell them apart, so they stay as hard negatives.
            if not a.positions and pg_of[f] and pg_of[f] in used_pg:
                continue
            chosen.append(f); used_pg.add(pg_of[f])
            if len(chosen) == B:
                break
        pairs = [random.sample(by_fn[f], 2) for f in chosen]
        negs = [] if a.no_near_miss else [random.choice(mutants_of[f]) for f in chosen if f in mutants_of]

        idx = [p[0] for p in pairs] + [p[1] for p in pairs] + negs     # one big batch = one pass
        z = F.normalize(batch_embed(idx, tr_graphs, tr_parsed), dim=1)
        n = len(pairs)
        za, zb, zn = z[:n], z[n:2 * n], (z[2 * n:] if negs else None)
        loss = (info_nce(za, zb, zn) + info_nce(zb, za, zn)) / 2

        opt.zero_grad()
        loss.backward()
        th.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())

        if step % 50 == 0:
            mins = (time.perf_counter() - t_start) / 60
            print(f'step {step}: loss {np.mean(losses[-50:]):.3f}  ({mins:.1f} min)')
        if step % a.eval_every == 0 or step == a.steps:
            hard, full, full_rule = score_val()
            mins = (time.perf_counter() - t_start) / 60
            tag = ''
            if pick(hard, full_rule) > best:
                best = pick(hard, full_rule)
                th.save(model.state_dict(), os.path.join(a.out_dir, 'best.pth'))
                tag = '  <- new best, saved'
            print(f"  VAL step {step}: hard Hit@1 {hard['hit1']:.4f} | full Hit@1 {full['hit1']:.4f} | "
                  f"full + rule {full_rule['hit1']:.4f}{tag}")
            logw.writerow([step, np.mean(losses[-a.eval_every:]),
                           hard['hit1'], hard['mrr'], full['hit1'], full['mrr'],
                           full_rule['hit1'], full_rule['mrr'], round(mins, 2)]); log.flush()

    th.save(model.state_dict(), os.path.join(a.out_dir, 'last.pth'))
    print(f"\ndone. best val {'hard Hit@1' if a.select == 'hard' else 'full Hit@1 with input-count rule'} = {best:.4f}")
    print(f"weights: {os.path.join(a.out_dir, 'best.pth')}   log: {log_path}")


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/FGNN2/src/fgnn2_pos.py
"""
fgnn2_pos.py -- builds FGNN2, with optional add-ons:
  positions (Experiment B): input i starts with  ones + pos_emb[i]  instead of plain ones.
                            pos_emb starts at zero, so before training it equals pretrained FGNN2.
  input-count rule (Experiment C): see add_input_count below.
"""
import torch as th
from FunctionConv import FuncConv


def make_model(hidden_dim=256, positions=False, max_inputs=256, weights=None, device='cpu'):
    model = FuncConv(hidden_dim, hidden_dim, flag_proj=False, flag_inv=True)
    if positions:
        model.pos_emb = th.nn.Embedding(max_inputs + 1, hidden_dim, padding_idx=0)
        th.nn.init.zeros_(model.pos_emb.weight)
    model = model.to(device)
    if weights:
        state = th.load(weights, map_location=device, weights_only=True)
        missing = [k for k in model.state_dict() if k not in state]
        if missing:
            print('loaded weights; new parts start fresh:', sorted({k.split('.')[0] for k in missing}))
        model.load_state_dict(state, strict=not missing)
    return model


def add_input_count(emb, num_inputs, weight=10.0, max_inputs=256):
    """Input-count rule: append a scaled one-hot of the number of inputs, so circuits with a
    different number of inputs can never look close (they can't be twins in this dataset)."""
    import numpy as np
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
    oh = np.zeros((len(emb), max_inputs + 1), dtype=emb.dtype)
    oh[np.arange(len(emb)), np.minimum(np.asarray(num_inputs), max_inputs)] = weight
    return np.hstack([emb, oh])


In [ ]:
%%writefile /content/strict_eval.py
"""
strict_eval.py -- "strict" grader: swapped-input copies count as WRONG answers.

The official evaluate.py removes swapped-input copies (same perm_group_id, different function)
before scoring. This grader keeps them in, so its full-search scores match exactly what
top20_cec.py shows. Ranking rule (shared with top20_cec.py): highest cosine similarity first,
ties broken by sample_id.

Hard lineups contain no swapped-input copies, so strict hard = official hard.
Only needs numpy.

Example:
    python strict_eval.py --data_root /content/circuitree_v2 --emb /content/expB_real_val.npz
    python strict_eval.py ... --query r029642     # also show one query, to compare with its top-20 table
"""
import argparse, csv, os, sys
import numpy as np

KS = (1, 5, 10, 20)


def add_input_count(E, num_inputs, weight=10.0, max_inputs=256):
    """Input-count rule (same as fgnn2_pos.add_input_count): circuits with a different number of
    inputs can never look close."""
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    oh = np.zeros((len(E), max_inputs + 1))
    oh[np.arange(len(E)), np.minimum(np.asarray(num_inputs), max_inputs)] = weight
    return np.hstack([E, oh])


def rank(sim, ids_arr):
    """Order used by both strict_eval.py and top20_cec.py: highest similarity first, ties by sample_id."""
    return np.lexsort((ids_arr, -sim))


def query_metrics(is_twin_ranked):
    pos = np.flatnonzero(is_twin_ranked)
    if len(pos) == 0:
        return None
    m = {'hit1': float(is_twin_ranked[0]), 'mrr': 1.0 / (pos[0] + 1)}
    for k in KS[1:]:
        m[f'r{k}'] = is_twin_ranked[:k].sum() / len(pos)          # share of all twins in the top k
    m['any20'] = float(pos[0] < 20)                                 # at least one twin in the top 20
    m['ap'] = float((np.arange(1, len(pos) + 1) / (pos + 1)).mean())
    return m


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data_root', required=True)
    ap.add_argument('--emb', required=True)
    ap.add_argument('--track', default='real')
    ap.add_argument('--split', default='val')
    ap.add_argument('--query', default=None, help='also print this one query (to check against top20_cec.py)')
    ap.add_argument('--input_count', action='store_true', help='apply the input-count rule first')
    a = ap.parse_args()

    sys.path.insert(0, a.data_root)
    import evaluate as ev

    with open(os.path.join(a.data_root, 'samples.csv'), newline='') as f:
        info = {r['sample_id']: r for r in csv.DictReader(f)}
    z = np.load(a.emb)
    ids = [str(s) for s in z['sample_ids']]
    ids_arr = np.array(ids)
    E = z['embeddings'].astype(np.float64)
    if a.input_count:
        E = add_input_count(E, [int(info[s]['num_inputs']) for s in ids])
    E /= np.linalg.norm(E, axis=1, keepdims=True) + 1e-12
    fn = np.array([info[s]['function_id'] for s in ids])
    idx = {s: i for i, s in enumerate(ids)}
    with open(os.path.join(a.data_root, 'eval', f'{a.track}_{a.split}_queries.csv'), newline='') as f:
        queries = [r['query_id'] for r in csv.DictReader(f)]

    res = []
    for q in queries:
        i = idx[q]
        sim = E @ E[i]
        order = rank(sim, ids_arr)
        order = order[order != i]                                   # not itself
        m = query_metrics(fn[order] == fn[i])
        if m:
            res.append(m)
            if q == a.query:
                print(f"query {q}: Hit@1 {m['hit1']:.0f} | MRR {m['mrr']:.3f} | "
                      f"twins in top 10 {m['r10']:.2f} | top 20 {m['r20']:.2f}\n")
    strict = {k: float(np.mean([r[k] for r in res])) for k in res[0]}

    emb = dict(zip(ids, E))
    off_full = ev.evaluate(emb, a.track, a.split, 'full', root=a.data_root)
    off_hard = ev.evaluate(emb, a.track, a.split, 'hard', root=a.data_root)

    print(f"{a.track} {a.split} | {os.path.basename(a.emb)}{' + input-count rule' if a.input_count else ''} | "
          f"{len(res)} full-search queries")
    print(f"{'':28}{'Hit@1':>7}{'MRR':>7}{'top5':>7}{'top10':>7}{'top20':>7}{'>=1 in 20':>11}{'AP':>7}")
    print(f"{'full, STRICT (copies wrong)':28}{strict['hit1']:7.3f}{strict['mrr']:7.3f}{strict['r5']:7.3f}"
          f"{strict['r10']:7.3f}{strict['r20']:7.3f}{strict['any20']:11.3f}{strict['ap']:7.3f}")
    print(f"{'full, official (copies out)':28}{off_full['hit1']:7.3f}{off_full['mrr']:7.3f}{off_full['r5']:7.3f}"
          f"{off_full['r10']:7.3f}{'-':>7}{'-':>11}{off_full['ap']:7.3f}")
    print(f"{'hard (same either way)':28}{off_hard['hit1']:7.3f}{off_hard['mrr']:7.3f}{off_hard['r5']:7.3f}"
          f"{off_hard['r10']:7.3f}{'-':>7}{'-':>11}{off_hard['ap']:7.3f}")
    print('\ntopK = share of ALL of a query\'s twins that land in the top K (averaged over queries)')


if __name__ == '__main__':
    main()


## 3. Fingerprints + scores (real track, val split)

First the **pretrained** FGNN2. Then an **untrained** (random) FGNN2, to see how much the training helps.

To beat on real / val / hard Hit@1: random 0.35, size_stats 0.48, **wl_kernel 0.66**, simulation 0.96 (answer key).

In [ ]:
# FGNN2 makes a fingerprint for all 8,727 val circuits
!cd /content/FGNN2/src && /content/fgnn_env/bin/python embed_circuitree.py --data_root /content/circuitree_v2 --track real --split val --weights ../checkpoints/example/weight.pth --flag_inv --out /content/fgnn2_real_val.npz

In [ ]:
# the grader scores them (first line = hard, second line = full)
!cd /content/circuitree_v2 && python evaluate.py --embeddings /content/fgnn2_real_val.npz --track real --split val

In [ ]:
# same thing with an untrained FGNN2 (no --weights)
!cd /content/FGNN2/src && /content/fgnn_env/bin/python embed_circuitree.py --data_root /content/circuitree_v2 --track real --split val --flag_inv --out /content/fgnn2_random_real_val.npz
!cd /content/circuitree_v2 && python evaluate.py --embeddings /content/fgnn2_random_real_val.npz --track real --split val

## 4. Top 20 + CEC

Picks a random query circuit, lists FGNN2's 20 most similar circuits, and checks each with ABC's `cec`.
Add `--query r027344` (any sample ID) to choose the query yourself, or `--seed 1`, `--seed 2`, ... for other random ones.
The table is also saved as `top20.csv`.

In [ ]:
!python /content/top20_cec.py --data_root /content/circuitree_v2 --emb /content/fgnn2_real_val.npz --track real --split val --k 20 --out /content/top20.csv

## 5. Experiment A: fine-tune FGNN2 on the train split

Same model, just more practice on ~35,000 real training circuits, starting from the pretrained weights.
Every 250 steps it scores **val** and keeps the best weights. Test is never used.

Results are saved to Google Drive (`Circuitree/expA`), so they survive if Colab disconnects.

**5a is a quick test drive** (100 steps) to check that training runs and to see how long a step takes.
Only run 5b after 5a works.

In [ ]:
# 5a: test drive -- 100 steps (should start at val hard Hit@1 = 0.7799, same as Section 3)
!cd /content/FGNN2/src && /content/fgnn_env/bin/python finetune_fgnn2.py --data_root /content/circuitree_v2 --weights ../checkpoints/example/weight.pth --steps 100 --eval_every 50 --out_dir /content/expA_test

In [ ]:
# 5b: the real run -- 2000 steps
!cd /content/FGNN2/src && /content/fgnn_env/bin/python finetune_fgnn2.py --data_root /content/circuitree_v2 --weights ../checkpoints/example/weight.pth --steps 2000 --out_dir /content/drive/MyDrive/Circuitree/expA

In [ ]:
# 5c: fingerprints + official scores with the best fine-tuned weights, then top 20 + CEC on the same query as before
!cd /content/FGNN2/src && /content/fgnn_env/bin/python embed_circuitree.py --data_root /content/circuitree_v2 --track real --split val --weights /content/drive/MyDrive/Circuitree/expA/best.pth --flag_inv --out /content/expA_real_val.npz
!cd /content/circuitree_v2 && python evaluate.py --embeddings /content/expA_real_val.npz --track real --split val
!python /content/top20_cec.py --data_root /content/circuitree_v2 --emb /content/expA_real_val.npz --track real --split val --query r029642 --k 20 --out /content/top20_expA.csv

## 6. Experiment B: input positions + fine-tune

Same as Experiment A, with one change: each input (x0, x1, ...) gets its own learned starting fingerprint,
so FGNN2 can finally tell inputs apart. The position vectors start at zero, so at step 0 the model is
exactly the pretrained FGNN2 (val hard Hit@1 should again be 0.7799). Starts from the **pretrained** weights,
same 2000 steps as A, so the only difference from A is the positions.

**6a is a test drive** (100 steps). Only run 6b after 6a works.

In [ ]:
# 6a: test drive -- 100 steps
!cd /content/FGNN2/src && /content/fgnn_env/bin/python finetune_fgnn2.py --data_root /content/circuitree_v2 --weights ../checkpoints/example/weight.pth --positions --steps 100 --eval_every 50 --out_dir /content/expB_test

In [ ]:
# 6b: the real run -- 2000 steps, saved to Google Drive
!cd /content/FGNN2/src && /content/fgnn_env/bin/python finetune_fgnn2.py --data_root /content/circuitree_v2 --weights ../checkpoints/example/weight.pth --positions --steps 2000 --out_dir /content/drive/MyDrive/Circuitree/expB

In [ ]:
# 6c: official scores with the best Experiment B weights, then top 20 + CEC on the same query (r029642)
!cd /content/FGNN2/src && /content/fgnn_env/bin/python embed_circuitree.py --data_root /content/circuitree_v2 --track real --split val --weights /content/drive/MyDrive/Circuitree/expB/best.pth --positions --flag_inv --out /content/expB_real_val.npz
!cd /content/circuitree_v2 && python evaluate.py --embeddings /content/expB_real_val.npz --track real --split val
!python /content/top20_cec.py --data_root /content/circuitree_v2 --emb /content/expB_real_val.npz --track real --split val --query r029642 --k 20 --out /content/drive/MyDrive/Circuitree/expB/top20_expB.csv

## 7. Strict scores: swapped-input copies count as WRONG

The official grader removes swapped-input copies before scoring. `strict_eval.py` keeps them in and
counts them as wrong answers, so its full-search scores match the top-20 + CEC table exactly
(both use the same ranking rule). Hard lineups have no swapped-input copies, so hard is the same either way.

**top10 / top20** = share of ALL of a query's twins that land in the top 10 / 20 (what the lead cares about most).

Works on CPU too (no GPU needed). 8a makes fingerprints for all 4 models (a few minutes on CPU), 8b scores them.

**Exp C (final model) = Exp B weights + the input-count rule**, so it is the "B" row below (all rows use `--input_count`).

In [ ]:
# 7a: fingerprints for pretrained, Exp A, Exp B (plain, no input-count rule baked in)
W = '/content/drive/MyDrive/Circuitree'
!cd /content/FGNN2/src && /content/fgnn_env/bin/python embed_circuitree.py --data_root /content/circuitree_v2 --track real --split val --weights ../checkpoints/example/weight.pth --flag_inv --out /content/pre_val.npz
!cd /content/FGNN2/src && /content/fgnn_env/bin/python embed_circuitree.py --data_root /content/circuitree_v2 --track real --split val --weights {W}/expA/best.pth --flag_inv --out /content/A_val.npz
!cd /content/FGNN2/src && /content/fgnn_env/bin/python embed_circuitree.py --data_root /content/circuitree_v2 --track real --split val --weights {W}/expB/best.pth --positions --flag_inv --out /content/B_val.npz

In [ ]:
# 7b: strict + official scores for every model, with the input-count rule
for m in ['pre', 'A', 'B']:   # B + --input_count = Exp C (final)
    !python /content/strict_eval.py --data_root /content/circuitree_v2 --emb /content/{m}_val.npz --input_count
    print()

In [ ]:
# 7c: check that table and scores agree -- one query, same model, same rule
!python /content/strict_eval.py --data_root /content/circuitree_v2 --emb /content/B_val.npz --input_count --query r029642
!python /content/top20_cec.py --data_root /content/circuitree_v2 --emb /content/B_val.npz --input_count --query r029642 --k 20